# 🧪 Syndicate 3.0: Phase 6.5 Playroom (The Assertion Ledger)

Welcome to the **Agentic Evaluation Discovery Lab**. 

In the real world, building an agent is 10% prompt engineering and 90% evaluation. If you rely on 'vibes-based' manual testing (asking the model a few questions and checking if they look right), your system will break the moment you update a prompt, change parameters, or update the model version.

In this playroom, you will learn to build a scientific, deterministic evaluation framework using **pure Python primitives**. You will master validation logic, run-time trajectory logging, and strict assertions to prove your agent is reliable before it hits production.

---

## 📐 6.5.1: JSON Schema Conformance Tests
**Goal:** Build a strict, zero-dependency validation engine to assert that a model's proposed tool-calls conform to your function parameter schemas.

### 🧪 Discovery Lab: The Black-Box Tool-Call Trap

#### 1. The Naive Attempt (Raw Parsing)
Suppose we have a tool `get_weather` that requires a string parameter `city`. The model outputs a proposed tool-call dictionary. Let's parse it naively:

In [ ]:
# A proposed tool-call from a faulty or malicious model run
proposed_call = {
    "name": "get_weather",
    "arguments": {
        "city": 12345 # OOPS! This should be a string, not an integer!
    }
}

# Naive extraction and execution:
def query_database(city: str):
    return f"Weather database queried for: {city.upper()}"

try:
    # Attempting to run with model parameters directly
    result = query_database(proposed_call["arguments"]["city"])
    print(result)
except AttributeError as e:
    print(f"CRASHED at runtime: {e}")

#### 2. The Logical Necessity
The runtime crashed because we passed an integer `12345` to a function expecting a string, triggering an `AttributeError` when we called `.upper()`. 

To prevent this, we must build a **validation boundary** between the probabilistic model output and our deterministic code execution. We need a function that evaluates parameters against a target schema contract (asserting types and checking required parameters) *before* execution.

#### 3. The Rosetta Stone
Let's write a schema checker using pure Python. We will define a target schema contract as a dictionary, and build a validator that returns a list of conformance errors.

In [ ]:
from typing import Dict, Any, List

WEATHER_SCHEMA = {
    "required": ["city"],
    "properties": {
        "city": {"type": str},
        "days": {"type": int}
    }
}

def validate_json_schema(arguments: Dict[str, Any], schema: Dict[str, Any]) -> List[str]:
    errors = []
    
    # Step 1: Verify all required keys are present
    for req_key in schema.get("required", []):
        if req_key not in arguments:
            errors.append(f"Missing required parameter: {req_key}")
            
    # Step 2: Validate types of provided parameters
    for param, value in arguments.items():
        if param in schema["properties"]:
            expected_type = schema["properties"][param]["type"]
            if not isinstance(value, expected_type):
                errors.append(
                    f"Parameter '{param}' has invalid type. "
                    f"Expected {expected_type.__name__}, got {type(value).__name__}"
                )
        else:
            errors.append(f"Unexpected parameter: {param}")
            
    return errors

# Test our Rosetta Stone on the bad proposed call:
errs = validate_json_schema(proposed_call["arguments"], WEATHER_SCHEMA)
print(f"Validation Errors Caught: {errs}")
assert len(errs) == 1, "Should have caught exactly 1 error."

### 🚀 Active Lab Exercise: Enforcing Parameter Boundaries
**Your Task:** Write a robust schema validator that handles complex schemas containing type check definitions, multiple required keys, and custom value constraints.

Implement the function `validate_tool_arguments(arguments: dict, schema: dict) -> list[str]` which checks:
1. Presence of all required keys.
2. Correct type matching for parameters (`str`, `int`, `float`, `bool`).
3. Value constraints such as minimum value validation if `min_val` is present in a property's schema.

In [ ]:
def validate_tool_arguments(arguments: dict, schema: dict) -> list[str]:
    errors = []
    
    # TODO: Implement complete JSON schema validation logic
    # Check requirements, type check, and custom min_val constraint if it exists in property rules.
    # Your code here:

    
    return errors

# Target Schema for Tool: "send_payment"
PAYMENT_SCHEMA = {
    "required": ["recipient_id", "amount"],
    "properties": {
        "recipient_id": {"type": str},
        "amount": {"type": float, "min_val": 0.01},
        "memo": {"type": str}
    }
}

# Verification cases:
bad_call_1 = {"amount": 50.0} # Missing required 'recipient_id'
bad_call_2 = {"recipient_id": "user_123", "amount": "fifty dollars"} # Invalid type for amount
bad_call_3 = {"recipient_id": "user_123", "amount": -10.50} # Violation of min_val limit

# Add tests inside playroom to guide student
print("Ensure your validate_tool_arguments function passes these checks!")

---
## 📈 6.5.2: Trajectory Assertion Harness
**Goal:** Assert that an agent transitions through correct states and executes tools in the expected sequence for a specific intent.

### 🧪 Discovery Lab: The Wandering Agent Loop

#### 1. The Naive Attempt (Inspect the Final State)
Suppose we have an agent executing a database query, extracting data, and then saving it. We want to verify it completed correctly. Naively, we only check if the database got updated:

In [ ]:
# The final data store
database = {"records": []}

# Suppose a messy agent loop executed and updated the database, but did it by bypassing authorization steps?
database["records"].append("private_record") # Record was successfully inserted!

# Vibes-based final inspection:
assert len(database["records"]) > 0, "Verification failed!"
print("Looks good to me! Database contains records!")

# FAILURE: We have absolutely no idea HOW it got there. Did it check auth? Did it write logs?

#### 2. The Logical Necessity
Checking only the final state is a high-risk failure mode. In security-sensitive or high-reliability agentic flows, we must verify the **trajectory (execution path)**. We need a way to log state transitions and tool-calls in sequence at runtime, and write assertions against this historical trace.

#### 3. The Rosetta Stone
Let's build a simple, lightweight Trajectory Ledger using a pure Python class. It records actions sequentially and provides path checks.

In [ ]:
class TrajectoryLedger:
    def __init__(self):
        self.history: List[str] = []
        
    def record_step(self, state_name: str):
        self.history.append(state_name)
        
    def assert_transition_path(self, expected_path: List[str]):
        assert self.history == expected_path, (
            f"Trajectory Deviation! Expected: {expected_path}, but got: {self.history}"
        )

# Testing our ledger:
ledger = TrajectoryLedger()
ledger.record_step("AUTHENTICATE")
ledger.record_step("DB_SEARCH")
ledger.record_step("FORMAT_OUTPUT")

# Verify that execution followed standard protocols:
ledger.assert_transition_path(["AUTHENTICATE", "DB_SEARCH", "FORMAT_OUTPUT"])
print("Success! Execution trajectory verified perfectly.")

### 🚀 Active Lab Exercise: The Trajectory Auditor
**Your Task:** Write a trajectory checking algorithm that analyzes logs of transitions and verifies that no invalid state jumps occurred, and that critical check states were never bypassed.

Implement the function `audit_trajectory(log: list[str], illegal_transitions: list[tuple[str, str]], mandatory_checkpoints: list[str]) -> list[str]` which returns a list of audit failures. 

Rules:
1. If a checkpoint inside `mandatory_checkpoints` was not visited, return a failure: `"Missing mandatory checkpoint: [name]"`.
2. If the transition sequence contains an illegal jump `(state_A, state_B)` (e.g. going directly from `START` to `WRITE_DATABASE` without passing `AUTHENTICATE`), return a failure: `"Illegal transition: [state_A] -> [state_B]"`.

In [ ]:
def audit_trajectory(
    log: list[str], 
    illegal_transitions: list[tuple[str, str]], 
    mandatory_checkpoints: list[str]
) -> list[str]:
    failures = []
    
    # TODO: Implement the audit checks based on the rules
    # Your code here:

    
    return failures

# Audit settings:
ILLEGAL = [("START", "WRITE_DB"), ("START", "EXECUTE_COMMAND")]
MANDATORY = ["AUTH_CHECK", "INPUT_SANITY"]

# Test logs:
clean_log = ["START", "INPUT_SANITY", "AUTH_CHECK", "WRITE_DB", "END"]
malicious_log = ["START", "WRITE_DB", "END"] # Bypassed checkpoints, illegal jump!
incomplete_log = ["START", "INPUT_SANITY", "WRITE_DB", "END"] # Missing AUTH_CHECK

print("Audit trajectory testing functions loaded. Run audits to verify your code!")

---
## 🧪 6.5.3: Deterministic Run Verification
**Goal:** Build a regression suite with mock model inputs to verify routing transitions deterministically across prompt adjustments.

### 🧪 Discovery Lab: The Floating Prompt Bug

#### 1. The Naive Attempt (Live Testing)
When tweaking an agent's prompt, you might run the live model 3 times and see it work. 

*Why it fails:* LLMs are non-deterministic. If your prompt tweak caused a 2% degradation, you will never notice it with 3 manual live tests. But in production, with 10,000 users, that 2% drift means hundreds of crashed loops.

#### 2. The Logical Necessity
To test agentic architecture deterministically, we must separate the **Model Generator** from the **Agent Router Engine**. By replacing the live model endpoint with a deterministic **Mock Generator** that plays predefined responses, we can verify that our state machine code, tool routers, and parsers handle every conceivable output configuration (valid tool calls, empty responses, malformed JSON, and formatting variations) without drift.

#### 3. The Rosetta Stone
Let's write a mock test harness that runs a simulation using a predefined registry of model response outcomes, proving the agent handles edge cases consistently.

In [ ]:
# Predefined list of mock responses we want to test our agent loop against
MOCK_OUTCOMES = {
    "bad_format": "Error: JSON malformed here { city: London }",
    "valid_call": '{"name": "get_weather", "arguments": {"city": "London"}}',
    "empty": ""
}

class MockAgentEngine:
    def __init__(self, outcome_mode: str):
        self.mode = outcome_mode
        self.log = []
        
    def run_inference(self) -> str:
        return MOCK_OUTCOMES[self.mode]
        
    def run_cycle(self):
        self.log.append("START")
        raw_res = self.run_inference()
        self.log.append("INFERENCE_DONE")
        
        try:
            if not raw_res:
                self.log.append("HANDLE_EMPTY")
            elif raw_res.startswith("Error"):
                self.log.append("HANDLE_FORMAT_ERROR")
            else:
                self.log.append("EXECUTE_TOOL")
        except Exception:
            self.log.append("CRASH")
            
        self.log.append("END")

# Deterministically test if our empty recovery branch works:
engine = MockAgentEngine("empty")
engine.run_cycle()
print(f"Sequence Completed: {engine.log}")
assert "HANDLE_EMPTY" in engine.log, "Failed to handle empty completion!"
assert "CRASH" not in engine.log, "Engine crashed during recovery!"

### 🚀 Active Lab Exercise: The Regression Suite
**Your Task:** Build a regression suite that validates that an agent successfully routes distinct simulated conversations to the correct states without throwing exceptions.

Implement the agent loop verification runner `run_regression_suite(agent_class, test_cases: list[dict]) -> dict[str, str]` which routes cases and returns a dict mapping `case_name` to 'PASS' or 'FAIL'. Ensure the engine is robust to crashes.

In [ ]:
# TODO: Build the complete regression runner and assert conformance across all cases.
def run_regression_suite(agent_class, test_cases: list[dict]) -> dict[str, str]:
    results = {}
    
    # For each test case:
    # 1. Instantiate the agent with case variables
    # 2. Run the agent execution cycle
    # 3. Audit the final state and trajectory ledger history
    # 4. Bind "PASS" or "FAIL" in the results
    # Your code here:

    
    return results

# Simple testing mock structure
class SimulatedAgent:
    def __init__(self, response: str):
        self.response = response
        self.trajectory = []
        
    def execute(self):
        self.trajectory.append("START")
        if "tool_call" in self.response:
            self.trajectory.append("DISPATCH")
        elif "error" in self.response:
            self.trajectory.append("ERROR_HANDLER")
        else:
            self.trajectory.append("STANDARD_CHAT")
        self.trajectory.append("END")

REGRESSION_CASES = [
    {"name": "normal_chat", "response": "hello there", "expected_path": ["START", "STANDARD_CHAT", "END"]},
    {"name": "tool_request", "response": "tool_call_weather", "expected_path": ["START", "DISPATCH", "END"]},
    {"name": "bad_completion", "response": "error_500", "expected_path": ["START", "ERROR_HANDLER", "END"]}
]

print("Success! Predefined regression setup prepared.")

---
## 🏁 Phase 6.5 Completed!
You have successfully mastered agentic evaluation! You have moved away from vibes-based prompt hacks to strict, scientifically verified agent architecture. 

In the next phase, you will use these assertions to build **Phase 7: Pure-Python State Machines**!